In [1]:
"""
stage_5.py -- Stage 5 (iteration 2): 5c2-raw hazard model, manifest-driven.

Frozen model contract (5c2-raw): grammar (bsince + ewm{2,6,18} per class, age, tod)
+ causal z-scored values @ {t-1, t-2, t-3}: signed & magnitude of every stream's
source column, leg amplitude, raw body signed & magnitude. Sign convention
confirming-positive (value * leg_dir).

Manifest-driven (iteration 2):
  - Fork axes (frame, session window, stream set) are READ from the Stage-0
    manifest, never redeclared here. Dropping a stream in Stage 0 (e.g. no-TICK)
    propagates automatically: grammar classes AND z-value channels both track the
    manifest stream set.
  - tod is derived from clock time (session_start), resolution-independent.
  - The manifest is baked into the model bundle so the booster carries its own
    contract; the worker asserts against it and prints it at startup.

Naming (iteration 2):
  SOURCE_PATH / src : the "source" oscillator file (HA OHLC + JMA + TICK + derivs),
                      Stage-0's input; rawer than bars/events. src is the primary
                      data frame, augmented in-place with raw body columns.
  RAW_FILE / raw1   : pure MNQ raw OHLC (transient; merged into src, then unused).
"""

import json
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from scipy.signal import lfilter
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, roc_auc_score

from common import Featurizer, load_manifest, _expanding_z, _welford_check

In [2]:
# ---------------------------------------------------------------- CONFIG (per-run, explicit)
FRAME = 3
STAGE0_TAG = 'mnq-TICK-9-12am'

MANIFEST_PATH = f"stage-0/manifest_{STAGE0_TAG}_{FRAME}s.json"
BARS_PATH = f"stage-0/bars_{STAGE0_TAG}_{FRAME}s.pqt"
EVENTS_PATH = f"stage-0/events_{STAGE0_TAG}_{FRAME}s.pqt"

ITER_DIR = "."                                   # iteration-2 root (encapsulated)
OUT_DIR = "stage-5"

VALID_FROM = "2025-07-01"
TRAIN_END = "2025-12-31"
TEST_FROM = "2026-01-01"

# frozen 5c2 architecture constants (NOT fork axes -- stay in code)
TAUS = (2.0, 6.0, 18.0)
VALUE_LAGS = (1, 2)                               # t-2, t-3 (t-1 shift is implicit)
ZWARM = 20
TOD_BIN_MIN = 30
BODY_OPEN_COL = "Open"
BODY_CLOSE_COL = "Last"
BODY_TAG = "ha"

LGBM_PARAMS = dict(
    objective="binary", metric="binary_logloss", learning_rate=0.05,
    num_leaves=127, min_data_in_leaf=1000, feature_fraction=0.9,
    bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
    num_threads=16, verbosity=-1,
)
NUM_ROUNDS = 8000
EARLY_STOP = 200
STAGE5_ANCHOR = 0.27402
# ----------------------------------------------------------------

In [3]:
# ---------------------------------------------------------------- grammar features
def build_grammar_features(fz, date_from=None, date_to=None):
    blocks = []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        n = S["n"]
        cols = []
        for c in fz.classes:
            P = S["P"][c]
            ind = np.diff(P).astype(np.float64)
            occ = np.where(ind > 0, np.arange(n), -1)
            last = np.maximum.accumulate(occ)
            lastm1 = np.concatenate(([-1], last[:-1]))
            bsince = np.where(lastm1 >= 0, np.arange(n) - lastm1, np.arange(n) + 1)
            cols.append(bsince[t])
            x = np.concatenate(([0.0], ind[:-1]))
            for tau in TAUS:
                a = np.exp(-1.0 / tau)
                s = lfilter([a], [1.0, -a], x)
                cols.append(s[t])
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)
        age = np.where(lt >= 0, t - lt, t + 1)
        cols.append(age)
        cols.append(S["tod"][t].astype(np.float64))
        blocks.append(np.stack(cols, 1).astype(np.float32))
    return np.concatenate(blocks), fz.grammar_names

In [4]:
# ---------------------------------------------------------------- value features
def value_base_names(manifest):
    """Value channels derived from the manifest stream set (source columns)."""
    cols = manifest["_stream_cols"]
    names = ([f"z_{c}_signed" for c in cols]
             + [f"z_{c}_mag" for c in cols]
             + ["z_leg_amp", f"z_body_{BODY_TAG}_signed", f"z_body_{BODY_TAG}_mag"])
    return cols, names


def build_value_features(fz, src, date_from=None, date_to=None):
    cols, base = value_base_names(fz.manifest)
    names = base + [f"{nm}_lag{L}" for L in VALUE_LAGS for nm in base]
    sv = src.set_index("timestamp")
    blocks = []
    for S in fz._selected(date_from, date_to):
        ts = pd.DatetimeIndex(S["timestamp"])
        r = sv.reindex(ts)
        leg_dir = S["leg_dir"]
        feats = []
        for c in cols:                                             # signed per stream col
            feats.append(_expanding_z(r[c].to_numpy(np.float64) * leg_dir, ZWARM))
        for c in cols:                                             # magnitude per stream col
            feats.append(_expanding_z(np.abs(r[c].to_numpy(np.float64)), ZWARM))
        feats.append(_expanding_z(np.abs(r["JMA"].to_numpy(np.float64) - S["leg_start_jma"]), ZWARM))
        bo = r[BODY_OPEN_COL].to_numpy(np.float64)
        bc = r[BODY_CLOSE_COL].to_numpy(np.float64)
        feats.append(_expanding_z((bc - bo) * leg_dir, ZWARM))     # confirming-positive
        feats.append(_expanding_z(np.abs(bc - bo), ZWARM))

        M = np.stack(feats, 1)
        M = np.concatenate([np.zeros((1, M.shape[1])), M[:-1]], 0)          # t-1 shift
        lagged = [M]
        for L in VALUE_LAGS:
            lagged.append(np.concatenate([np.zeros((L, M.shape[1])), M[:-L]], 0))
        M = np.concatenate(lagged, 1)
        t = np.nonzero(~S["warm"])[0]
        Mt = M[t]
        Mt[(t < ZWARM + max(VALUE_LAGS))] = 0.0
        blocks.append(Mt.astype(np.float32))
    return np.concatenate(blocks), names


def build_X(fz, src, date_from=None, date_to=None):
    Xg, gn = build_grammar_features(fz, date_from, date_to)
    Xv, vn = build_value_features(fz, src, date_from, date_to)
    assert len(Xg) == len(Xv), (len(Xg), len(Xv))
    return np.hstack([Xg, Xv]), gn + vn


def build_meta(fz, date_from=None, date_to=None):
    bi, ts, tg, dt = [], [], [], []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        bi.append(S["bar_index"][t])
        ts.append(S["timestamp"][t])
        tg.append(S["tgt"][t])
        dt.append(np.full(len(t), str(S["sess"])))
    return pd.DataFrame({"bar_index": np.concatenate(bi),
                         "timestamp": np.concatenate(ts),
                         "is_target": np.concatenate(tg),
                         "date": np.concatenate(dt)})

In [5]:
# ---------------------------------------------------------------- train / eval
def train(fz, src, train_end, valid_from):
    X, names = build_X(fz, src, None, train_end)
    meta = build_meta(fz, None, train_end)
    y = meta["is_target"].to_numpy().astype(np.int8)
    va = (meta["date"] >= valid_from).to_numpy()
    tr = ~va
    dtr = lgb.Dataset(X[tr], label=y[tr], feature_name=names)
    dva = lgb.Dataset(X[va], label=y[va], reference=dtr)
    booster = lgb.train(LGBM_PARAMS, dtr, num_boost_round=NUM_ROUNDS,
                        valid_sets=[dva], valid_names=["valid"],
                        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False),
                                   lgb.log_evaluation(200)])
    p_va = booster.predict(X[va], num_iteration=booster.best_iteration)
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_va, y[va])
    print(json.dumps(dict(n_train=int(tr.sum()), n_valid=int(va.sum()),
                          best_iteration=int(booster.best_iteration),
                          valid_logloss_cal=float(log_loss(y[va], iso.predict(p_va)))),
                     indent=2))
    imp = pd.DataFrame({"feature": names,
                        "gain": booster.feature_importance("gain")}
                       ).sort_values("gain", ascending=False)
    print(imp.head(20).to_string(index=False))
    return dict(booster=booster, iso=iso, feature_names=names,
                valid_from=valid_from, train_end=train_end,
                manifest=fz.manifest, tag=STAGE0_TAG, importance=imp)


def evaluate(fz, src, model, start, end=None, anchor=STAGE5_ANCHOR):
    X, _ = build_X(fz, src, start, end)
    meta = build_meta(fz, start, end)
    y = meta["is_target"].to_numpy().astype(np.int8)
    p = model["booster"].predict(X, num_iteration=model["booster"].best_iteration)
    p_cal = model["iso"].predict(p)
    ll_cal = log_loss(y, p_cal)
    ll_const = log_loss(y, np.full_like(p, y.mean(), dtype=np.float64))
    print(json.dumps(dict(n_rows=int(len(y)), holdout_logloss_cal=float(ll_cal),
                          holdout_logloss_const=float(ll_const),
                          skill=float(1 - ll_cal / ll_const),
                          auc=float(roc_auc_score(y, p)),
                          anchor=anchor, delta=float(ll_cal - anchor)), indent=2))
    out = meta[["bar_index", "timestamp", "is_target"]].copy()
    out["p"] = p.astype(np.float32)
    out["p_cal"] = p_cal.astype(np.float32)
    tbl = out.assign(bin=pd.qcut(out["p_cal"], 10, duplicates="drop")).groupby(
        "bin", observed=True).agg(mean_p=("p_cal", "mean"),
                                  realized=("is_target", "mean"), n=("p_cal", "size"))
    print(tbl.to_string())
    return out

In [6]:
# ---------------------------------------------------------------- run
manifest = load_manifest(MANIFEST_PATH,TOD_BIN_MIN)

SOURCE_PATH = manifest["source_file"]
RAW_FILE = manifest["raw_file"]

assert STAGE0_TAG == manifest["stage0_tag"]

bars = pd.read_parquet(BARS_PATH)
events = pd.read_parquet(EVENTS_PATH)

sess_lo = pd.Timestamp(manifest["session_start"]).time()
sess_hi = pd.Timestamp(manifest["session_end"]).time()

src = pd.read_parquet(SOURCE_PATH)
src = src[(src["timestamp"].dt.time >= sess_lo) & (src["timestamp"].dt.time < sess_hi)]

raw1 = pd.read_parquet(RAW_FILE)
raw1 = raw1[(raw1["timestamp"].dt.time >= sess_lo) & (raw1["timestamp"].dt.time < sess_hi)]
raw1 = raw1.rename(columns={"Open": "rawOpen", "High": "rawHigh", "Low": "rawLow", "Last": "rawLast"})

src = src.merge(raw1[["timestamp", "rawOpen", "rawHigh", "rawLow", "rawLast"]], on="timestamp", how="left")

###  assert day start exists in src and raw ###
src_min_time = src["timestamp"].dt.time.min()
print(f'SRC MIN TIME: {src_min_time}')
assert sess_lo == src_min_time

assert src[["rawOpen", "rawLast"]].notna().all().all(), "raw OHLC has gaps vs source timestamps"

SRC MIN TIME: 09:00:00


In [7]:
print('--------------------- !! VERIFY !! ---------------------')
print(f'FRAME: {FRAME}sec, STAGE0_TAG: {STAGE0_TAG}, BODY_TAG: {BODY_TAG}')
print('----------------------- MANIFEST -----------------------')
print(json.dumps({k: v for k, v in manifest.items() if not k.startswith("_")}, indent=2))
print('--------------------------------------------------------')

#

fz = Featurizer(bars, events, manifest, TOD_BIN_MIN, TAUS)
#augment_featurizer(fz, bars)

S0 = fz.sessions[len(fz.sessions) // 2]
xchk = src.set_index("timestamp").reindex(pd.DatetimeIndex(S0["timestamp"]))["jmaD1"].to_numpy(np.float64)
print("welford max abs diff:", _welford_check(xchk, ZWARM))

model = train(fz, src, TRAIN_END, VALID_FROM)
pred = evaluate(fz, src, model, TEST_FROM)

print(f"-------------- {STAGE0_TAG} {FRAME}s --------------")
joblib_file = f"{OUT_DIR}/model_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.joblib"
importance_file = f"{OUT_DIR}/importance_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.csv"
pred_file = f"{OUT_DIR}/pred_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.pqt"

joblib.dump({k: v for k, v in model.items() if k != "importance"}, joblib_file)
model["importance"].to_csv(importance_file, index=False)
pred.to_parquet(pred_file, index=False)

print(f'    joblib_file: {joblib_file}')
print(f'importance_file: {importance_file}')
print(f'      pred_file: {pred_file}')

#

z = pred.timestamp >= TEST_FROM
print("holdout-window logloss:", log_loss(pred.is_target[z], pred.p_cal[z]))

--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: mnq-TICK-9-12am, BODY_TAG: ha
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    },
    {
      "name": "TICK_D2",
      "column": "tickJmaD2"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 4120088,
  "n_sessions": 1145,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-08 00:00:00",
  "created": "2026-07-15T18:34:42.402584",
  "stage0_tag": "mnq-TICK-9-12am"
}
--------------------------------------------------------
welford max abs diff: 4.440892098500626e-15
[200]	valid's binary_logloss: 0.13388

In [8]:
# visual check
print(f"-------------- bars --------------")
print(bars.head())
print(f"\n-------------- events --------------")
print(events.head())
print(f"\n-------------- model --------------")
print(model)
print(f"\n-------------- pred --------------")
print(pred.head())

-------------- bars --------------
   bar_index           timestamp       date           jma        d1  \
0          0 2022-01-03 09:00:00 2022-01-03  16384.462891 -1.261719   
1          1 2022-01-03 09:00:03 2022-01-03  16382.718750 -2.554688   
2          2 2022-01-03 09:00:06 2022-01-03  16380.848633 -3.614258   
3          3 2022-01-03 09:00:09 2022-01-03  16378.701172 -4.017578   
4          4 2022-01-03 09:00:12 2022-01-03  16376.974609 -3.874023   

   jma_leg_dir  leg_id  leg_age   leg_amp  is_target  warm  
0            0       0        0  0.000000      False  True  
1           -1       0        1  1.744141      False  True  
2           -1       0        2  3.614258      False  True  
3           -1       0        3  5.761719      False  True  
4           -1       0        4  7.488281      False  True  

-------------- events --------------
        date   stream  event_bar  extremum_bar           timestamp  \
0 2022-01-03   MNQ_D1         12            11 2022-01-03 09:00:

In [9]:
'''
------------ !! VERIFY !! ------------
FRAME: 3sec, STAGE0_TAG: mnq-TICK-9-12am
-------------- MANIFEST --------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    },
    {
      "name": "TICK_D2",
      "column": "tickJmaD2"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 4120088,
  "n_sessions": 1145,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-08 00:00:00",
  "created": "2026-07-15T18:34:42.402584",
  "stage0_tag": "mnq-TICK-9-12am"
}
SRC MIN TIME: 09:00:00

---

welford max abs diff: 4.440892098500626e-15
[200]	valid's binary_logloss: 0.133504
[400]	valid's binary_logloss: 0.129178
[600]	valid's binary_logloss: 0.127939
[800]	valid's binary_logloss: 0.127486
[1000]	valid's binary_logloss: 0.127265
[1200]	valid's binary_logloss: 0.127122
[1400]	valid's binary_logloss: 0.127028
[1600]	valid's binary_logloss: 0.127006
[1800]	valid's binary_logloss: 0.126983
[2000]	valid's binary_logloss: 0.127005
{
  "n_train": 3175513,
  "n_valid": 452137,
  "best_iteration": 1878,
  "valid_logloss_cal": 0.12647040290118156
}
                feature         gain
      z_body_raw_signed 2.040263e+06
            z_jmaD1_mag 1.395213e+06
         z_jmaD2_signed 1.296681e+06
         z_jmaD1_signed 1.078511e+06
 z_body_raw_signed_lag1 1.070985e+06
         z_body_raw_mag 9.690214e+05
       z_jmaD1_mag_lag1 5.213821e+05
    z_jmaD1_signed_lag1 3.407416e+05
    z_body_raw_mag_lag1 3.316400e+05
      MNQ_D1|opp|bsince 2.965744e+05
     MNQ_D1|conf|bsince 2.783907e+05
 z_body_raw_signed_lag2 2.633367e+05
MNQ_JMA_SELF|all|bsince 2.370317e+05
            z_jmaD2_mag 2.264946e+05
       z_jmaD1_mag_lag2 1.873037e+05
        MNQ_D1|opp|ewm2 1.552500e+05
  MNQ_JMA_SELF|all|ewm2 1.188696e+05
                    tod 1.172857e+05
    z_jmaD1_signed_lag2 1.127774e+05
    z_jmaD2_signed_lag1 9.817903e+04
{
  "n_rows": 477398,
  "holdout_logloss_cal": 0.12311719559596355,
  "holdout_logloss_const": 0.3334844163241089,
  "skill": 0.6308157455960172,
  "auc": 0.9715751679821629,
  "anchor": 0.27402,
  "delta": -0.15090280440403642
}
                        mean_p  realized      n
bin                                            
(-0.001, 6.61e-05]    0.000037  0.000109  54947
(6.61e-05, 0.000196]  0.000191  0.000272  69915
(0.000196, 0.000635]  0.000614  0.000685  55477
(0.000635, 0.000907]  0.000843  0.001233  13790
(0.000907, 0.00203]   0.001462  0.001287  48189
(0.00203, 0.00711]    0.004717  0.003669  49603
(0.00711, 0.0245]     0.014311  0.013615  48842
(0.0245, 0.0894]      0.053702  0.050657  41159
(0.0894, 0.402]       0.214108  0.216849  48347
(0.402, 1.0]          0.742915  0.764391  47129
-------------- mnq-TICK-9-12am 3s --------------
    joblib_file: stage-5/model_mnq-TICK-9-12am_3s.joblib
importance_file: stage-5/importance_mnq-TICK-9-12am_3s.csv
      pred_file: stage-5/pred_mnq-TICK-9-12am_3s.pqt
holdout-window logloss: 0.12299088388681412
'''

'\n------------ !! VERIFY !! ------------\nFRAME: 3sec, STAGE0_TAG: mnq-TICK-9-12am\n-------------- MANIFEST --------------\n{\n  "frame_seconds": 3,\n  "session_start": "09:00",\n  "session_end": "12:00",\n  "warmup_bars": 10,\n  "streams": [\n    {\n      "name": "MNQ_D1",\n      "column": "jmaD1"\n    },\n    {\n      "name": "MNQ_D2",\n      "column": "jmaD2"\n    },\n    {\n      "name": "TICK_D1",\n      "column": "tickJmaD1"\n    },\n    {\n      "name": "TICK_D2",\n      "column": "tickJmaD2"\n    }\n  ],\n  "self_stream": "MNQ_JMA_SELF",\n  "source_file": "data/mnq-tick-full-3sec.pqt",\n  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",\n  "n_bars": 4120088,\n  "n_sessions": 1145,\n  "date_min": "2022-01-03 00:00:00",\n  "date_max": "2026-07-08 00:00:00",\n  "created": "2026-07-15T18:34:42.402584",\n  "stage0_tag": "mnq-TICK-9-12am"\n}\nSRC MIN TIME: 09:00:00\n\n---\n\nwelford max abs diff: 4.440892098500626e-15\n[200]\tvalid\'s binary_logloss: 0.133504\n[400]\tvalid\'s binary_loglos

In [ ]:
'''
--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: mnq-TICK-9-12am, BODY_TAG: ha
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    },
    {
      "name": "TICK_D2",
      "column": "tickJmaD2"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 4120088,
  "n_sessions": 1145,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-08 00:00:00",
  "created": "2026-07-15T18:34:42.402584",
  "stage0_tag": "mnq-TICK-9-12am"
}
--------------------------------------------------------
welford max abs diff: 4.440892098500626e-15
[200]	valid's binary_logloss: 0.133884
[400]	valid's binary_logloss: 0.132125
[600]	valid's binary_logloss: 0.131762
[800]	valid's binary_logloss: 0.131611
[1000]	valid's binary_logloss: 0.131523
[1200]	valid's binary_logloss: 0.131558
{
  "n_train": 3175513,
  "n_valid": 452137,
  "best_iteration": 1000,
  "valid_logloss_cal": 0.13111789595471177
}
              feature         gain
     z_body_ha_signed 5.925607e+06
       z_jmaD1_signed 1.447015e+06
        z_body_ha_mag 1.148606e+06
       z_jmaD2_signed 5.952857e+05
          z_jmaD2_mag 2.506119e+05
z_body_ha_signed_lag1 2.298040e+05
          z_jmaD1_mag 2.123958e+05
z_body_ha_signed_lag2 1.712491e+05
                  tod 1.118566e+05
   z_body_ha_mag_lag1 1.116254e+05
   z_body_ha_mag_lag2 9.854100e+04
  z_jmaD2_signed_lag1 8.666702e+04
     z_jmaD1_mag_lag2 6.932561e+04
MNQ_JMA_SELF|all|ewm2 6.640025e+04
  z_jmaD1_signed_lag2 6.508477e+04
     z_jmaD1_mag_lag1 4.597998e+04
      MNQ_D1|opp|ewm2 4.571720e+04
   z_tickJmaD2_signed 4.291389e+04
     z_jmaD2_mag_lag2 4.203009e+04
  z_jmaD1_signed_lag1 4.139849e+04
{
  "n_rows": 477398,
  "holdout_logloss_cal": 0.12994797440416128,
  "holdout_logloss_const": 0.3334844163241089,
  "skill": 0.6103326930939207,
  "auc": 0.9678507302260672,
  "anchor": 0.27402,
  "delta": -0.1440720255958387
}
                        mean_p  realized      n
bin                                            
(-0.001, 3.32e-05]    0.000017  0.000084  59391
(3.32e-05, 0.000377]  0.000308  0.000369  70382
(0.000377, 0.00059]   0.000503  0.000468  23500
(0.00059, 0.001]      0.000993  0.001499  54032
(0.001, 0.00303]      0.002559  0.002378  38687
(0.00303, 0.00869]    0.005668  0.005086  41485
(0.00869, 0.0248]     0.015518  0.015230  48982
(0.0248, 0.108]       0.060325  0.056866  49344
(0.108, 0.365]        0.223204  0.228121  44187
(0.365, 1.0]          0.735613  0.749346  47408
-------------- mnq-TICK-9-12am 3s --------------
    joblib_file: stage-5/model_ha_mnq-TICK-9-12am_3s.joblib
importance_file: stage-5/importance_ha_mnq-TICK-9-12am_3s.csv
      pred_file: stage-5/pred_ha_mnq-TICK-9-12am_3s.pqt
holdout-window logloss: 0.12986378371715546
'''